In [1]:
# globally useful libraries / namespaces for this notebook
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.animation
import matplotlib.pyplot as plt

# notebook wide settings for plotting visualizations
plt.rcParams['figure.figsize'] = (10, 8) # set default figure size, 10in by 8in

# Chapter 2: The Mathematical Building Blocks of Neural Networks

Supporting materials for:

Chollet (2021). *Deep Learning with Python*. 2nd ed. Manning Publications Co.
[Amazon](https://www.amazon.com/Learning-Python-Second-Fran%C3%A7ois-Chollet/dp/1617296864/ref=sr_1_1?crid=32NFM2SBCJVQQ)

Understanding deep learning requires familiarity with many simple mathematical
concepts: tensors, tensor operations, differentiation, gradient descent, and so on.
Out goal in this notebook/chapter is to build up your intuition about these mathematical
concepts without getting over technical.

One note about the term tensors.  **Tensor** is a mathematical term that simply can be used to refer
to a matrix of values, no matter how many dimensions.  Thus a 1-dimensional vector or a 2-dimensional
matrix are both *Tensors*, as well as 3-dimensional or higher matrices.  Vectorized programming
could also be referred to as tensor programming, but the term vectorized programming was coined
before the term tensor was in common use.  Another difference is that the term Tensor programming
and Tensor libraries have come to mean the types of vectorized or tensor operations we usually think of, but
it also includes libraries that can automatically differentiate such tensors with respect to
a loss function.  We will get to these concepts in a bit more detail below.



## Note: Installing Keras/TensorFlow

For this textbook we are using the `TensorFlow` library and mostly accessing the `Keras`
neural network and deep learning API through the `TensorFlow`.  You will need to have
a recent version of `Tensorflow/Keras` installed on your machine and available to your
iPython kernel to run this notebook.  At the time of creation of this notebook, 
`TensorFlow/Keras` is no longer supported directly by the `Conda` package manager.  You
should be doing a `pip install` command of `TensorFlow` to correctly get the 
most recent `TensorFlow` libraries with the `Keras` API available:

```
$ python3 -m pip install 'tensorflow'
```

If you have a suitable Nvidia GPU processor available on your your machine, you can get the `cuda` libraries
setup and installed for `TensorFlow` by doing:

```
python3 -m pip install 'tensorflow[and-cuda]'
```

See the official [Install TensorFlow 2](https://www.tensorflow.org/install) for more detailed information about correctly
getting `TensorFlow/Keras` installed on your system. [Install TensorFlow with pip](https://www.tensorflow.org/install/pip) 
has information on installing cuda/Nvidia.


# 2.5 Looking Back at our First Example

The first example at start of chapter in section 2.1 used the keras high level api of tensorflow to 
specify a fully connected network with a dense layer followed by the output activation layer.

In this section to wrap things up, the author digs deeper into tensorflow and computation graphs, showing a bit
of a hand-built implementation of the high level operations here using lower more basic level tensorflow
concepts.

## 2.5.1 Reimplementing our First Example from Scratch

A simple Dense class.

In [2]:
import tensorflow as tf

class NaiveDense:
    """Simple example of how Dense layers in tf.keras can be implemented
    using tensorflow variables to create tensorflow compute graphs.
    
    This is a class that we can use to instantiate a layer with some
    particular number of inputs and outputs, that maintains a weight
    matrix W and bias vector b in order to implement forward passes
    to calculate outputs given inputs, and to allow for calculation
    of gradients so that weights W and biases b can be updated by
    optimization to improve model performance iteratively.
    """
    
    def __init__(self, input_size, output_size, activation):
        """Class constructor, construct initial weight matrix W and
        bias matrix b with small random values and 0 values respectively.
        """
        self.activation = activation
        
        # create a matrix W (model weights) of shape (input_size, output_size)
        # initialized with random values
        w_shape = (input_size, output_size)
        w_initial_value = tf.random.uniform(w_shape, minval=0, maxval=1e-1)
        self.W = tf.Variable(w_initial_value)
        
        # create a vector, b (bias vector) initialized with zeros
        b_shape = (output_size,)
        b_initial_value = tf.zeros(b_shape)
        self.b = tf.Variable(b_initial_value)
        
        
    def __call__(self, inputs):
        """Overload call operation on class to implement forward pass, calculating layer
        outputs given some inputs to the layer.
        """
        return self.activation(tf.matmul(inputs, self.W) + self.b)
    
    @property
    def weights(self):
        """Convenience method for retrieving the layer's weights
        """
        return [self.W, self.b]
    

A simple sequential class.

In [3]:
class NaiveSequential:
    """Example implementation of a Keras sequential model, which collects and composes a sequence of
    layers, like the NaiveDense layer, together into a multi-layer model. 
    """
    
    def __init__(self, layers):
        """Sequential model constructor, we are just given a list/tuple of layers which the model
        manages and uses.
        """
        self.layers = layers
        
    def __call__(self, inputs):
        """Overload call operation for Sequential model.  Basically perform a forward pass through
        all layers, transforming inputs to a layer to outputs, which are then feed as inputs to next
        layer in the sequential model, until final layer is reached.  Final outputs are
        returned from calling this sequential model.
        """
        x = inputs
        for layer in self.layers:
            # call forward pass on current layer using inputs from previous layer/loop
            x = layer(x)
            
        # return outputs calculated from final layer
        return x
    
    @property
    def weights(self):
        """Convenience method for gathering weights of all layers as a list
        """
        weights = []
        for layer in self.layers:
            weights += layer.weights
        return weights


Using the `NaiveDense` class and the `NaiveSequential` class, we can create a mock Keras model

In [4]:
model = NaiveSequential([
    # hidden layer is shape (784 inputs, 512 hidden unit outputs), and uses relu activation function
    NaiveDense(input_size=28 * 28, output_size=512, activation=tf.nn.relu),
    
    # output layer is shape (512 inputs, 10 outputs) and uses a softmax activation function
    NaiveDense(input_size=512, output_size=10, activation=tf.nn.softmax)
])

assert len(model.weights) == 4
model.weights

[<tf.Variable 'Variable:0' shape=(784, 512) dtype=float32, numpy=
 array([[0.00622309, 0.06917816, 0.07916007, ..., 0.07419064, 0.05472859,
         0.02316389],
        [0.05138167, 0.01235101, 0.01275291, ..., 0.03314196, 0.06756023,
         0.0379207 ],
        [0.0720889 , 0.05786748, 0.00500393, ..., 0.05585662, 0.03435646,
         0.06673812],
        ...,
        [0.00146394, 0.05938093, 0.01557509, ..., 0.06114818, 0.0986205 ,
         0.07164158],
        [0.01214249, 0.0157342 , 0.02260352, ..., 0.00433824, 0.00397139,
         0.03823861],
        [0.08120566, 0.07580151, 0.08058868, ..., 0.09883735, 0.02065393,
         0.07227977]], dtype=float32)>,
 <tf.Variable 'Variable:0' shape=(512,) dtype=float32, numpy=
 array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0

Next we need a way to iterate over MNIST data in mini-batches.  In Python this is easy using slicing to slice up inputs into
batch-sized chunks.

In [5]:
import math

class BatchGenerator:
    """Convenience class that implements an iterator like interface to break up training inputs into
    batch-sized samples for batch training
    """
    
    def __init__(self, images, labels, batch_size=128):
        """Constructor is given full training data (images) and labels.  Defaults to
        batch sizes of 128 if not specified in construction of the generator.
        """
        # number of image samples and number of image labels must match
        assert len(images) == len(labels)
        
        # initial take first batch starting at index 0
        self.index = 0
        
        self.images = images
        self.labels = labels
        self.batch_size = batch_size
        
        # calculate number of batches to complete a full training cycle, we round up so last batch might
        # end up with fewer training samples that all of the preceding ones
        self.num_batches = math.ceil(len(images) / batch_size)
        
    def next(self):
        """Generator/iterator, calculate and return next batch of input samples and output
        labels for batch training.
        """
        # slice out the next batch of inputs and outputs
        images = self.images[self.index : self.index + self.batch_size]
        labels = self.labels[self.index : self.index + self.batch_size]
        
        # update index for next batch iteration
        self.index += self.batch_size
        
        return images, labels

## 2.5.2 Running one Training Step

The most difficult part is the "training step".  We need to update the weights after the model runs.  We need to perform the following
steps to train a model:

1. Compute the predictions (output) of the model for the images in the batch
2. Compute the loss value for those predictions, given the actual labels.
3. Compute the gradient of the loss with regard to the model's weights.
4. Move the weights by a small amount (learning rate) in the direction opposite to the gradient.

To compute the gradient, we will use the TensorFlow `GradientTape` object.

In [6]:
def one_training_step(model, images_batch, labels_batch):
    """Function to perform a single training step on a single batch for
    our example NaiveModel.  We expect the current model, and
    a batch of images and labels as inputs.  The model performs
    the steps 1-4 described above.
    """
    with tf.GradientTape() as tape:
        # 1. forward pass, compute predictions of the model
        predictions = model(images_batch)

        # 2. compute loss, use crossentropy loss function, notice we get a loss for each batch
        # sample, and we then just average the losses over all samples to compute the
        # output layer loss
        per_sample_losses = tf.keras.losses.sparse_categorical_crossentropy(labels_batch, predictions)
        average_loss = tf.reduce_mean(per_sample_losses)
        #print(f"average loss {average_loss}")
        
        # 3. Compute the gradient of the loss with regard to the weights.  The output gradients is a list
        # where each entry corresponds to a weight from the model.weights list.
        gradients = tape.gradient(average_loss, model.weights)

        # 4. Update the weights using the gradient
        # this is actually a call to another function that applies the weight updates
        update_weights(gradients, model.weights)

        return average_loss

As you know, the purpose of the "weight update" step is to move the weights by "a bit" in a direction that will reduce
loss on this batch (e.g. batch gradient descent).  The magnitude of the weight movement is determined by a
`learning_rate` metaparameter, which is a parameter that is usually specified for the keras/optimizer.

The simplest way to implement this is to subtract `gradient * learning_rate` from each weight.

In [7]:
# might have to be modified to get training to converge, usually this would be another parameter specified for
# the model or optimizer
learning_rate = 0.0001

def update_weights(gradients, weights):
    """Perform weight update for gradient descent optimization using the calculated gradients of
    the current weights.
    """
    for g, w in zip(gradients, weights):
        # assign_sub is the equivalent of -= for TensorFlow variables.
        w.assign_sub(g * learning_rate)

In [8]:
# in practice you wouldn't want to update weights by hand, but use an optimizer from tf/keras library, e.g.
from tensorflow.keras import optimizers

optimizer = optimizers.SGD(learning_rate=1e-3)

def update_weights_alternative(gradients, weights):
    optimizer.apply_gradients(zip(gradients, weights))

Now that per-batch training step is ready, we can move on to implement an entire epoch of training.

## 2.5.3 The Full Training Loop

An epoch of training consists of repeating the training step for each batch in the training data.  This is what 1 epoch of training
is usually defined as.  We then will normally perform multiple epochs of training on the full batched training set until some stopping
criteria is reached (convergence of the loss function when it no longer changes by some threshold), or more simply just for some fixed
number of epochs.

In [9]:
def fit(model, images, labels, epochs, batch_size=128):
    """Perform batch gradient descent.  Perform the indicated number of epochs of training. For each epoch, 
    train on all of the batches of the training data.
    """
    # perform the indicated number of epochs of training
    for epoch_counter in range(epochs):
        print(f"Epoch {epoch_counter}")
        
        # Create a batch generator/iterator for this epoch of training
        batch_generator = BatchGenerator(images, labels)
        
        # iterate over all batches of the training data to perform 1 epoch of training
        for batch_counter in range(batch_generator.num_batches):
            images_batch, labels_batch = batch_generator.next()
            loss = one_training_step(model, images_batch, labels_batch)
            # display batch progress by reporting loss
            if batch_counter % 100 == 0:
                print(f"   loss at batch {batch_counter}: {loss:.2f}")

Lets test drive it.

In [10]:
from tensorflow.keras.datasets import mnist

# reload the mnist data again
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype("float32") / 255

In [11]:
# fit the model with 10 epochs of batch training
fit(model, train_images, train_labels, epochs=50, batch_size=128)

Epoch 0
   loss at batch 0: 6.03
   loss at batch 100: 2.30
   loss at batch 200: 2.29
   loss at batch 300: 2.31
   loss at batch 400: 2.26
Epoch 1
   loss at batch 0: 2.27
   loss at batch 100: 2.26
   loss at batch 200: 2.25
   loss at batch 300: 2.27
   loss at batch 400: 2.22
Epoch 2
   loss at batch 0: 2.22
   loss at batch 100: 2.22
   loss at batch 200: 2.21
   loss at batch 300: 2.22
   loss at batch 400: 2.18
Epoch 3
   loss at batch 0: 2.18
   loss at batch 100: 2.18
   loss at batch 200: 2.16
   loss at batch 300: 2.18
   loss at batch 400: 2.14
Epoch 4
   loss at batch 0: 2.14
   loss at batch 100: 2.15
   loss at batch 200: 2.12
   loss at batch 300: 2.14
   loss at batch 400: 2.10
Epoch 5
   loss at batch 0: 2.10
   loss at batch 100: 2.11
   loss at batch 200: 2.08
   loss at batch 300: 2.10
   loss at batch 400: 2.07
Epoch 6
   loss at batch 0: 2.06
   loss at batch 100: 2.07
   loss at batch 200: 2.04
   loss at batch 300: 2.07
   loss at batch 400: 2.03
Epoch 7
   lo

## 2.5.4 Evaluating the Model

We can evaluate the model by taking the `argmax` of its predictions over the test images and comparing to the expected labels.

In [12]:
predictions = model(test_images)

# convert tensorflow tensor/variable to a numpy array
predictions = predictions.numpy()

predicted_labels = np.argmax(predictions, axis=1)
matches = predicted_labels == test_labels
print(f"accuracy: {matches.mean():.2f}")

accuracy: 0.81


Currently getting an accuracy of 82% which is not great compared to the original keras example.  But loss is still going down so could/should
probably increase learning rate and or continue training to reduce loss further.